# 05 - Ablation Study

Evaluate the contribution of pathway vs. clinical features by training on three feature sets:
1. **Pathway Only** (8 features): 7 pathway scores + ratio
2. **Clinical Only** (9 features): encoded clinical variables
3. **Combined** (17 features): pathway + clinical

Each configuration uses identical 5-fold stratified CV on GSE96058.

**Expected results**:
- Pathway Only: EN 0.641, RF 0.645, GB 0.633
- Clinical Only: EN 0.847, RF 0.849, GB 0.851
- Combined: EN 0.855, RF 0.856, GB 0.827

**Requires**: Downloaded GSE96058 expression data.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np

from src.data_loader import load_clinical_data, load_gse96058_expression
from src.preprocessing import zscore_normalize, encode_clinical_features, filter_outcome
from src.features import compute_pathway_scores, add_ratio_features
from src.models import run_ablation

## 1. Prepare Features

In [ ]:
# Load data
gse_clin = load_clinical_data('../data/clinical/01_gse96058_clinical.csv')
gse_clin = filter_outcome(gse_clin)
gse_exp = load_gse96058_expression('../data/raw/GSE96058_gene_expression.csv')

# Align samples
common = list(set(gse_clin['sample_id']) & set(gse_exp['sample_id']))
gse_clin = gse_clin[gse_clin['sample_id'].isin(common)].sort_values('sample_id').reset_index(drop=True)
gse_exp = gse_exp[gse_exp['sample_id'].isin(common)].sort_values('sample_id').reset_index(drop=True)

# Pathway features
gse_exp_norm = zscore_normalize(gse_exp)
pathway_features = compute_pathway_scores(gse_exp_norm)
pathway_features = add_ratio_features(pathway_features)

# Clinical features
clinical_features = encode_clinical_features(gse_clin)

y = gse_clin['high_risk'].values

print(f"Pathway features: {pathway_features.shape[1]} columns")
print(f"Clinical features: {clinical_features.shape[1]} columns")
print(f"Samples: {len(y)}, High risk: {y.sum()}, Low risk: {(y == 0).sum()}")

## 2. Run Ablation Study

In [ ]:
print("Running ablation study (3 feature sets x 3 models x 5 folds)...\n")
ablation_results = run_ablation(pathway_features, clinical_features, y)

print("\n" + "=" * 70)
print("ABLATION STUDY RESULTS")
print("=" * 70)
for fs in ablation_results['Feature Set'].unique():
    print(f"\n{fs}:")
    subset = ablation_results[ablation_results['Feature Set'] == fs]
    for _, row in subset.iterrows():
        print(f"  {row['Model']:20s}  AUC: {row['AUC']:.3f} +/- {row['AUC_SD']:.3f}  Acc: {row['Accuracy']:.3f}")

## 3. Save Results

In [ ]:
ablation_results.to_csv('../results/ablation_results.csv', index=False)
ablation_results.to_csv('../data/processed/ablation_results.csv', index=False)
print("Saved ablation results to results/ablation_results.csv")
print("Saved ablation results to data/processed/ablation_results.csv")

## 4. Key Findings

- Clinical features alone provide strong predictive performance (~0.85 AUC)
- Pathway features alone achieve moderate performance (~0.64 AUC)
- Combined features slightly improve over clinical-only, demonstrating that pathway scores capture complementary biological information